In [1]:
import sys
sys.path.append('../Common')

import CommonMT5
import MetaTrader5 as mt5

In [2]:
def detectSignal(symbol, from_date, to_date, timeframe):

    import pandas as pd
    import plotly.graph_objects as go
    import redis
    import numpy as np
    import talib

    from datetime import datetime
    from statsmodels.tsa.arima.model import ARIMA

    # ##############################################Step 1: Lấy dữ liệu##############################################
    data = CommonMT5.CommonMT5.loaddataMT5_FromTo(symbol, from_date, to_date, timeframe)

    # ##############################################Step 2: Chiến lược##############################################  
    # Giả sử `data` là DataFrame chứa các cột Datetime, Open, High, Low, Close, Volume
    # Đặt chỉ mục DataFrame với cột 'Datetime' có dữ liệu kiểu datetime
    data.set_index('Datetime', inplace=True)

    # Tính Momentum
    data['Momentum'] = data['Close'].diff()

    # Tính RSI
    data['RSI'] = talib.RSI(data['Close'], timeperiod=14)

    # Chọn tham số ARIMA dựa trên AIC, BIC hoặc các tiêu chí khác (cần phải thực hiện trước) => Chay cell o tren
    model = ARIMA(data['Close'], order=(1, 0, 4))
    model_fit = model.fit()

    # Dự đoán giá
    data['Predicted_Close'] = model_fit.predict(start=0, end=len(data)-1, typ='levels')

    # Xác định tín hiệu mua và bán
    data['Buy_Signal'] = ((data['Predicted_Close'] > data['Close']) & (data['Momentum'] > -1) & (data['RSI'] < 80))
    data['Sell_Signal'] = ((data['Predicted_Close'] < data['Close']) & (data['Momentum'] < 2000) & (data['RSI'] > 30))

    # ##############################################Step 3: Đẩy dữ liệu qua Redis##############################################
    # Nếu có tín hiệu thì đẩy qua Redis
    # Datetime  Open    High    Low	Close   Volume  SMA short_ema   long_ema    MACD    Signal_Line Buy_Signal  Sell_Signal
    # Tạo kết nối Redis
    r = redis.Redis(host='localhost', port=6379, db=9)
    # Tạo hash key
    hash_key = 'OG_FX_Arima, Momentum, RSI_K13'
    last_record = data.iloc[-2]

    # Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
    if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True):
        for field, value in last_record.to_dict().items():
            # Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
            if isinstance(value, pd.Timestamp):
                value = value.isoformat()
            elif isinstance(value, (int, np.uint64)):
                value = str(value)
            r.hset(hash_key, field, value)  
            r.hset(hash_key, 'Symbol', symbol)
            r.hset(hash_key, 'Insertdate', datetime.now().isoformat())
        print(last_record)   
    else:
        print(last_record)   
        print('Không có tín hiệu!')   

In [3]:
from datetime import datetime, timedelta
import schedule
import time

def schedule_detectSignal():
    symbol = 'EURUSD.sml'
    from_date = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua
    to_date = (datetime.now() + timedelta(days=2)).strftime('%Y-%m-%d')
    timeframe = mt5.TIMEFRAME_M1
    detectSignal(symbol, from_date, to_date, timeframe)

In [4]:
# # Task chay: Viet theo While True
# # Danh sách các phút cụ thể bạn muốn hàm được chạy
# # run_minutes = [17, 27, 50, 2]
# run_minutes = list(range(0,60,1))
# last_run = None

# while True:
#     current_time = datetime.now()
#     current_minute = current_time.minute
#     # Kiểm tra xem phút hiện tại có nằm trong danh sách các phút cần chạy hàm không
#     if current_minute in run_minutes:
#         # Kiểm tra xem đã chạy hàm trong phút hiện tại hay chưa
#         last_run = getattr(schedule_detectSignal, 'last_run', None)
#         if last_run is None or last_run != current_minute:
#             schedule_detectSignal()
#             # Lưu lại phút cuối cùng mà hàm đã chạy
#             setattr(schedule_detectSignal, 'last_run', current_minute)   

#     # Nghỉ 1 giây trước khi kiểm tra lại
#     time.sleep(1)

In [ ]:
import schedule
schedule.every(1).minute.at(":00").do(schedule_detectSignal)
while True:
    schedule.run_pending()
    time.sleep(1)

c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization fai

Open                 1.16122
High                 1.16129
Low                  1.16114
Close                1.16118
Volume                   168
Momentum            -0.00003
RSI                75.822642
Predicted_Close     1.161205
Buy_Signal              True
Sell_Signal            False
Name: 2025-10-22 18:01:00, dtype: object


c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization fai

Open                 1.16122
High                 1.16129
Low                  1.16114
Close                1.16118
Volume                   168
Momentum            -0.00003
RSI                75.822642
Predicted_Close     1.161205
Buy_Signal              True
Sell_Signal            False
Name: 2025-10-22 18:01:00, dtype: object


c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization fai

...

Open                 1.16114
High                 1.16115
Low                  1.16098
Close                1.16099
Volume                   228
Momentum            -0.00015
RSI                58.591189
Predicted_Close     1.161139
Buy_Signal              True
Sell_Signal            False
Name: 2025-10-22 18:06:00, dtype: object


c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization fai

Open                 1.16101
High                 1.16121
Low                    1.161
Close                1.16116
Volume                   187
Momentum             0.00017
RSI                63.013481
Predicted_Close     1.160998
Buy_Signal             False
Sell_Signal             True
Name: 2025-10-22 18:08:00, dtype: object


c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization fai

Open                1.16117
High                1.16139
Low                 1.16108
Close               1.16125
Volume                  218
Momentum            0.00009
RSI                65.13627
Predicted_Close    1.161166
Buy_Signal            False
Sell_Signal            True
Name: 2025-10-22 18:09:00, dtype: object
